In [3]:
# @title QR Code Generator (High Res) { display-mode: "form" }

# --- 1. Install Necessary Libraries ---
# 'capture' hides the installation output to keep the notebook clean.
%%capture
!pip install qrcode[pil] Pillow requests

# --- 2. Imports ---
import qrcode
from PIL import Image, ImageDraw, ImageFont
import requests
import io
from IPython.display import display
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Warning: Not running in Google Colab. Automatic file download will not work.")

# ==========================================
# --- USER CONFIGURATION (EDIT HERE) ---
# ==========================================

# The URL or text needed for the QR Code
URL_DATA = "https://efrem.io/pcamp-2025" # @param {type:"string"}

# Short text to appear in the bottom-right corner
BRAND_TEXT = "efrem.io" # @param {type:"string"}

# Text Size (Percentage of total image height)
# 15 is a good balance. Increase to 20 or 25 for much larger text.
TEXT_SIZE_PERCENTAGE = 100 # @param {type:"integer", min:5, max:40}

# Colors (Use English names or hex codes, e.g., "#FF5733")
FILL_COLOR = "black" # Color of the QR modules and text @param {type:"string"}
BACK_COLOR = "white" # Background color @param {type:"string"}

# Resolution Control
# box_size defines how many pixels each "module" (square) has.
# 30 generates a very high-resolution image.
BOX_SIZE = 30 # @param {type:"integer", min:10, max:50}

# URL of a clean TrueType font (.ttf).
# Using Montserrat Bold from Google Fonts for a modern look.
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat-Bold.ttf"

# ==========================================
# --- HELPER FUNCTIONS ---
# ==========================================

def download_font(url):
    """Downloads a font from the web to use in Colab."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return io.BytesIO(response.content)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading font: {e}. Using default font (might look small).")
        return None

def generate_custom_qr(data, text, fill_color, back_color, box_size, font_url, text_pct):
    """
    Generates a high-res QR code with text in the bottom-right corner.
    """
    print("Generating QR Code... Please wait.")

    # 1. QR Code Configuration
    # Using ERROR_CORRECT_H (High) is crucial here.
    # It allows up to 30% of the code to be covered/damaged (by our text)
    # and still be scannable.
    qr = qrcode.QRCode(
        version=None, # Auto-detect size based on data
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=box_size,
        border=4, # Standard quiet zone
    )
    qr.add_data(data)
    qr.make(fit=True)

    # 2. Create Base Image
    img_qr = qr.make_image(fill_color=fill_color, back_color=back_color).convert('RGB')

    # If no text is provided, return the standard QR
    if not text:
        return img_qr

    # 3. Prepare to draw text
    draw = ImageDraw.Draw(img_qr)
    img_width, img_height = img_qr.size

    # Load Font
    font_file = download_font(font_url)
    if font_file:
        # Calculate font size dynamically based on the percentage user defined
        # Example: 0.15 * 1000px = 150px font size
        font_size = int(img_height * (text_pct / 100))
        # Ensure a minimum readable size
        font_size = max(font_size, 40)

        try:
            font = ImageFont.truetype(font_file, font_size)
        except Exception:
             print("Failed to load downloaded font. Using system default.")
             font = ImageFont.load_default()
    else:
        font = ImageFont.load_default()

    # 4. Calculate Text Position
    # textbbox returns (left, top, right, bottom) of the rendered text
    text_bbox = draw.textbbox((0, 0), text, font=font)
    text_width = text_bbox[2] - text_bbox[0]
    text_height = text_bbox[3] - text_bbox[1]

    # Padding from the bottom-right corner
    padding_x = box_size
    padding_y = box_size // 2

    # X Position: Image Width - Text Width - Padding
    pos_x = img_width - text_width - padding_x
    # Y Position: Image Height - Text Height - Padding
    pos_y = img_height - text_height - padding_y - (text_bbox[1] * 0.5)

    # 5. (Optional) Clear area behind text
    # This creates a "box" of the background color behind the text.
    # This makes the text strictly readable by erasing the QR pixels behind it.
    margin_box = box_size // 4
    draw.rectangle(
        [
            pos_x - margin_box,
            pos_y - margin_box,
            pos_x + text_width + margin_box,
            pos_y + text_height + margin_box
        ],
        fill=back_color
    )

    # 6. Draw the text
    draw.text((pos_x, pos_y), text, font=font, fill=fill_color)

    print(f"Success! Resolution: {img_width}x{img_height} pixels.")
    return img_qr

# ==========================================
# --- MAIN EXECUTION ---
# ==========================================

final_image = generate_custom_qr(
    URL_DATA,
    BRAND_TEXT,
    FILL_COLOR,
    BACK_COLOR,
    BOX_SIZE,
    FONT_URL,
    TEXT_SIZE_PERCENTAGE
)

# Display in Notebook
display(final_image)

# Save and Download
if IN_COLAB:
    file_name = "custom_qr_code.png"
    final_image.save(file_name)
    try:
        files.download(file_name)
        print(f"Downloading: {file_name}")
    except Exception as e:
        print(f"Auto-download failed. File saved as '{file_name}' in Colab files.")